In [7]:
# Origin Version
# Please note that this version is not optimized and does not contain other features
# It is only for reference. The full version is below.
import cv2
import numpy as np
import os

# Load video
video_path = "videos/video1.mp4"
cap = cv2.VideoCapture(video_path)

# Create output directory and prepare video writer
os.makedirs("output", exist_ok=True)  
fps = int(cap.get(cv2.CAP_PROP_FPS))  # Get frames per second of input video
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)) # Frame width
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) # Frame height
fourcc = cv2.VideoWriter_fourcc(*'mp4v') # Define codec for output video
out = cv2.VideoWriter("output/video_base.mp4", fourcc, fps, (width, height))

# Read the first frame and convert to grayscale
ret, prev_frame = cap.read()
prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

frame_count = 0
# Main processing loop
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    # Convert current frame to grayscale for optical flow calculation
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Optical Flow Calculation (Farneback)
    # Estimate motion between previous and current grayscale frames
    flow = cv2.calcOpticalFlowFarneback(prev_gray, gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
    # Convert flow vectors to magnitude and angle
    mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    
    # Define threshold for motion magnitude to consider
    motion_thresh = 1.5
    mask = (mag > motion_thresh).astype(np.uint8)

    # Find contours directly on thresholded magnitude map
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    for cnt in contours:
        # Draw even small contours
        x, y, w, h = cv2.boundingRect(cnt)
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 0, 255), 2)
        
    # Write processed frame to output video
    out.write(frame)
    # Update previous frame for next optical flow computation
    prev_gray = gray.copy()
    # Optional progress output
    frame_count += 1
    if frame_count % 30 == 0:
        print(f"Processed {frame_count} frames...")
        
# Release resources
cap.release()
out.release()
print("Tracking completed")


Processed 30 frames...
Processed 60 frames...
Processed 90 frames...
Processed 120 frames...
Processed 150 frames...
Processed 180 frames...
Processed 210 frames...
Processed 240 frames...
Processed 270 frames...
Processed 300 frames...
Processed 330 frames...
Processed 360 frames...
Tracking completed


In [5]:
#Full version
import cv2
import numpy as np
import os
import math

# Load input video
video_path = "videos/video1.mp4"
cap = cv2.VideoCapture(video_path)

# Set up output path and video writer
os.makedirs("output", exist_ok=True)
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter("output/video1.mp4", fourcc, fps, (width, height))

# Read the first frame and convert to grayscale for optical flow base
ret, prev_frame = cap.read()
prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
# Initialize HOG-based person detector
hog = cv2.HOGDescriptor()
hog.setSVMDetector(cv2.HOGDescriptor_getDefaultPeopleDetector())

# Tracking state containers
candidates = {}   # id -> (x, y, w, h, life, lost)
trajectories = {} # id -> [trajectory points]
next_id = 0       # for assigning unique object IDs 
# Please note that this id is only used for algorithmic tracking of objects and has no other purpose.
frame_count = 0   # Track frame number

MAX_LOST_FRAMES = 5   # Max allowed consecutive missing frames
MIN_CONFIRM_FRAMES = 3# Min lifespan before a target is considered real
MAX_TRAJ_LENGTH = 40  # how many past points to keep per object
REAL_HEIGHT_CM = 175  # average real-world height assumption for speed estimation
PREDICT_STEPS = 5     # how many future steps to predict in trajectory

# anti-noise / filtering parameters
FLOW_MAG_THRESHOLD = 2.5   # Threshold for optical flow magnitude (motion strength)      
MIN_HOG_HEIGHT = 200       # Ignore HOG boxes shorter than this (likely not people)
EDGE_MARGIN = 0            # Ignore boxes too close to image edges 
MATCH_DIST_THRESHOLD = 50  # Spatial distance threshold for tracking ID continuity      

# Estimate object speed in m/s based on image motion and pixel height
def estimate_speed(last_point, current_point, pixel_height, fps):
    if last_point is None or pixel_height == 0:
        return 0.0
    dx = current_point[0] - last_point[0]
    dy = current_point[1] - last_point[1]
    dist_pixel = math.sqrt(dx ** 2 + dy ** 2)
    ratio = REAL_HEIGHT_CM / pixel_height   # pixels -> cm
    speed_cm_per_s = dist_pixel * ratio * fps
    return round(speed_cm_per_s / 100.0, 1)    # pixels -> cm

# Predict future trajectory using polynomial fitting
def predict_trajectory(traj, steps=5):
    if len(traj) < 5:
        return []
    traj_np = np.array(traj, dtype=np.float32)
    x = np.arange(len(traj_np))
    x_pred = np.arange(len(traj_np), len(traj_np) + steps)

    try:
        fx = np.poly1d(np.polyfit(x, traj_np[:, 0], 2))
        fy = np.poly1d(np.polyfit(x, traj_np[:, 1], 2))
        pred_points = [(int(fx(i)), int(fy(i))) for i in x_pred]
        return pred_points
    except np.linalg.LinAlgError:
        return []

# Main processing loop
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    # Preprocessing: grayscale + blur to reduce noise
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (7, 7), 0)
    # Optical Flow (dense) from previous frame to current
    flow = cv2.calcOpticalFlowFarneback(prev_gray, gray, None, pyr_scale=0.5, levels=3, winsize=15, iterations=3, poly_n=5, poly_sigma=1.2, flags=0)
    mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    hog_rects, _ = hog.detectMultiScale(gray, winStride=(8, 8), padding=(8, 8), scale=1.05)

    # Collect initial raw detections from HOG with basic filtering
    raw_detections = []
    for (x, y, w, h) in hog_rects:
        # Skip detections that are too small or too close to the image border
        if h < MIN_HOG_HEIGHT or x < EDGE_MARGIN or y < EDGE_MARGIN or (x + w) > width - EDGE_MARGIN or (y + h) > height - EDGE_MARGIN:
            continue
        # Use optical flow magnitude to check if the region is truly moving
        region_mag = mag[y:y + h, x:x + w]
        motion_score = np.mean(region_mag)
        if motion_score > 1.0:
            raw_detections.append((x, y, w, h))
    
    # Filter out large boxes that completely contain smaller ones
    valid_detections = []
    for i, (x1, y1, w1, h1) in enumerate(raw_detections):
        keep = True
        for j, (x2, y2, w2, h2) in enumerate(raw_detections):
            if i != j:
                # If box i is fully inside box j keep if box i fully contains box j drop.
                if x2 >= x1 and y2 >= y1 and (x2 + w2) <= (x1 + w1) and (y2 + h2) <= (y1 + h1):
                    keep = False
                    break
        if keep:
            valid_detections.append((x1, y1, w1, h1))

    # Match valid detections to existing tracked candidates
    updated_ids = set()
    for (x, y, w, h) in valid_detections:
        cx, cy = x + w // 2, y + h // 2  # center of current detection
        matched = False
        # Try to match with existing candidates by proximity
        for obj_id, obj in candidates.items():
            ox, oy, ow, oh, life, lost = obj
            ocx, ocy = ox + ow // 2, oy + oh // 2 # center of existing object
            if abs(cx - ocx) < MATCH_DIST_THRESHOLD and abs(cy - ocy) < MATCH_DIST_THRESHOLD:
                # If matched, update the bounding box smoothly (linear interpolation)
                alpha = 0.6
                new_x = int(alpha * x + (1 - alpha) * ox)
                new_y = int(alpha * y + (1 - alpha) * oy)
                new_w = int(alpha * w + (1 - alpha) * ow)
                new_h = int(alpha * h + (1 - alpha) * oh)
                candidates[obj_id] = (new_x, new_y, new_w, new_h, life + 1, 0)
                # Append current center to trajectory
                trajectories[obj_id].append((new_x + new_w // 2, new_y + new_h // 2))
                # Limit trajectory history length
                if len(trajectories[obj_id]) > MAX_TRAJ_LENGTH:
                    trajectories[obj_id].pop(0)
                matched = True
                updated_ids.add(obj_id)
                break
        # If unmatched, register as a new object
        if not matched:
            candidates[next_id] = (x, y, w, h, 1, 0)
            trajectories[next_id] = [(x + w // 2, y + h // 2)]
            updated_ids.add(next_id)
            next_id += 1

    # Remove or update candidates that were not matched in this frame
    to_delete = []
    for obj_id in list(candidates.keys()):
        if obj_id not in updated_ids:
            x, y, w, h, life, lost = candidates[obj_id]
            lost += 1
            if lost > MAX_LOST_FRAMES:
                to_delete.append(obj_id)
            else:
                candidates[obj_id] = (x, y, w, h, life, lost)
    # delete lost objects and their trajectories
    for obj_id in to_delete:
        del candidates[obj_id]
        if obj_id in trajectories:
            del trajectories[obj_id]

    # Draw results, bounding boxes, speed and predicted trajectory
    for obj_id, (x, y, w, h, life, lost) in candidates.items():
        if life >= MIN_CONFIRM_FRAMES:
            center = (x + w // 2, y + h // 2)
            traj = trajectories.get(obj_id, [])
            last = traj[-2] if len(traj) > 1 else None
            speed = estimate_speed(last, center, h, fps)

            # Draw bounding box and speed
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 0, 255), 4)
            cv2.putText(frame, f"{speed} m/s", (x, y - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

            # draw historical + predicted trajectory
            if len(traj) >= 2:
                predicted = predict_trajectory(traj, steps=PREDICT_STEPS)
                full_traj = traj + predicted
                pts = np.array(full_traj, np.int32).reshape((-1, 1, 2))
                cv2.polylines(frame, [pts], isClosed=False, color=(0, 255, 0), thickness=2)
            
                # Draw arrow at the tip of predicted trajectory
                if len(predicted) >= 2:
                    p1 = predicted[-2]
                    p2 = predicted[-1]
                    cv2.arrowedLine(frame, p1, p2, (0, 165, 255), 4, tipLength=0.9)

    # Write annotated frame to output
    out.write(frame)
    prev_gray = gray.copy()
    frame_count += 1
    if frame_count % 30 == 0:
        print(f"Processed {frame_count} frames...")

# Release all resources
cap.release()
out.release()
print("Tracking completed")

Processed 30 frames...
Processed 60 frames...
Processed 90 frames...
Processed 120 frames...
Processed 150 frames...
Processed 180 frames...
Processed 210 frames...
Processed 240 frames...
Processed 270 frames...
Processed 300 frames...
Processed 330 frames...
Processed 360 frames...
Tracking completed
